# Climate change in ESA listing rules

**Question.** When the Fish and Wildlife Service lists a species under the Endangered Species Act, does it say climate change is a reason? How has that changed since the 1990s, and how does it compare with Delach et al. (2019)?

## Setup

Libraries, folders and a polite `fetch()` that pauses between requests.

In [1]:
import os, re, json, time, html, difflib
from pathlib import Path
from typing import Literal
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import requests
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

DATA, RESULTS = Path("data"), Path("results")
for folder in (DATA / "text", DATA / "mods", RESULTS):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

def fetch(url, params=None, tries=5):
    """GET with a polite pause between requests and a few retries."""
    for attempt in range(tries):
        r = requests.get(url, params=params, headers={"User-Agent": "espm157-text-as-data (student project)"}, timeout=120)
        if r.status_code == 200:
            time.sleep(0.5)
            return r
        time.sleep(5 * (attempt + 1))
    r.raise_for_status()

## Corpus

Every document the Federal Register API tags as an FWS rule, one request per year, 1994–2026 (cached in `data/`). ESA rules have titles starting "Endangered and Threatened Wildlife and Plants".

In [2]:
FIELDS = ["document_number", "title", "publication_date", "action", "abstract", "citation", "page_length",
          "agencies", "html_url", "raw_text_url"]

def fws_rules(year):
    """Every document the Federal Register API tags as a Fish and Wildlife Service rule in one year."""
    params = {"conditions[agencies][]": "fish-and-wildlife-service", "conditions[type][]": "RULE",
              "conditions[publication_date][gte]": f"{year}-01-01", "conditions[publication_date][lte]": f"{year}-12-31",
              "per_page": 1000, "fields[]": FIELDS, "order": "oldest"}
    rows = fetch("https://www.federalregister.gov/api/v1/documents.json", params).json().get("results", [])
    for row in rows:
        row["agency_names"] = "; ".join(a.get("name", "") for a in row.pop("agencies"))
    return pd.DataFrame(rows)

fr_path = DATA / "fr_fws_rules.parquet"
if not fr_path.exists():
    pd.concat([fws_rules(y) for y in range(1994, 2027)]).to_parquet(fr_path, index=False)
fr = pd.read_parquet(fr_path)
is_fws = fr.title.str.match(r"(?i)\s*Endangered and Threatened Wildlife and Plants")
is_joint = fr.title.str.match(r"(?i)\s*Endangered and Threatened Species") & fr.agency_names.str.contains("National Oceanic")
esa = fr[is_fws | is_joint].assign(source="Federal Register API")
per_year = fr.publication_date.str[:4].value_counts().sort_index()
print(f"{len(fr):,} FWS rules, {len(esa)} of them under the ESA ({is_joint.sum()} joint rules with NOAA Fisheries)")
print(f"FWS rules per year: {per_year['1994']} in 1994, then {per_year.iloc[1:].min()} to {per_year.iloc[1:].max()} a year; "
      f"the newest was published {fr.publication_date.max()}")

1,602 FWS rules, 965 of them under the ESA (3 joint rules with NOAA Fisheries)
FWS rules per year: 1 in 1994, then 14 to 71 a year; the newest was published 2026-09-25


Full text. `raw_text_url` returns a "Request Access" page for scripted downloads, so the text comes from GPO's GovInfo instead (cached in `data/text/`).

In [3]:
def govinfo_text(document_number, publication_date):
    """Plain text of one Federal Register document from GPO's GovInfo, cached in data/text/."""
    path = DATA / "text" / f"{document_number}.txt"
    if not path.exists():
        url = f"https://www.govinfo.gov/content/pkg/FR-{publication_date}/html/{document_number}.htm"
        body = fetch(url).text.split("<pre>", 1)[1].rsplit("</pre>", 1)[0]
        path.write_text(html.unescape(re.sub(r"<[^>]+>", "", body)))
    return path.read_text()

texts = {d: govinfo_text(d, p) for d, p in zip(esa.document_number, esa.publication_date)}
print(f"{len(texts)} rules, {sum(map(len, texts.values())) / 1e6:.0f} million characters")

965 rules, 161 million characters


A listing rule is a final rule whose regulatory text *adds* entries to the List in 50 CFR 17.11(h) or 17.12(h). Rules that only revise an entry are status re-determinations, kept out of the corpus.

In [4]:
def regulatory_text(text):
    """The amendatory part of a rule (after the last 'List of Subjects' or 'PART 17' heading)."""
    text = re.sub(r"\s+", " ", text)
    heads = list(re.finditer(r"List of Subjects in 50 CFR Parts? 17", text, re.I)) or \
            list(re.finditer(r"PART 17\s*-+\s*(?:\[?AMENDED|ENDANGERED)", text))
    return text[heads[-1].start():] if heads else text[int(len(text) * 0.85):]

def list_actions(text):
    """Which verbs a rule applies to the Lists at 50 CFR 17.11(h) and 17.12(h): add, revise, remove."""
    reg, actions = regulatory_text(text), set()
    for m in re.finditer(r"17\.1[12]\b", reg):
        before, after = reg[max(0, m.start() - 160):m.start()], reg[m.end():m.end() + 400]
        if re.search(r"appears? in the (?:table|list)|same alphabetical order|species appears", before, re.I):
            continue  # a critical-habitat instruction that only points back at the List
        stop = re.search(r"Sec\.\s*17\.(?!1[12]\b)\d+|\(h\)\s*\*\s*\*\s*\*", after)
        window = before[-120:] + " " + (after[:stop.start()] if stop else after)
        for verb, pattern in [("add", r"\badd(?:ing|ed)?\b|the following (?:is|are) added"),
                              ("revise", r"\brevis(?:e|ing|ed)\b"), ("remove", r"\bremov(?:e|ing|ed)\b")]:
            if re.search(pattern, window, re.I):
                actions.add(verb)
    return actions

STATUS = r"(?:endangered|threatened)(?: species)? status|status for|determin\w*(?: of)? (?:endangered|threatened|status)|determined? .{0,80}to be|\bto list\b|\blisting\b|as (?:endangered|threatened)|addition of|\badding\b|species status"
OTHER_ACTION = (r"reclassif|downlist|uplist|delist|\bremov|experimental population|taxonom|reinstat|technical correction"
                r"|regulations? (?:for|pertaining)|listing endangered and threatened species and designating|change in listing status"
                r"|prohibitions|manatee|designated ports|address")  # the last four only sort 10 rules that aren't listings into this group:
                                                                    # captive-bred antelope, manatee protection areas, ports for plants, an address change

def classify_rule(row, text):
    """Label one ESA rule: a new listing, or the reason it is not one."""
    title = re.sub(r"(?i)^\s*Endangered and Threatened (?:Wildlife and Plants|Species)[;:,]?\s*", "", row.title)
    action = row.action.lower() if isinstance(row.action, str) else ""
    if not action.startswith(("final rule", "direct final rule")) or re.search(r"correction|delay|technical|availability|reopen|re-open|stay|withdraw", action) or re.match(r"C\d-", row.document_number):
        return "not a final listing-type rule"
    actions = list_actions(text)
    adds, status = "add" in actions, bool(re.search(STATUS, title, re.I))
    if re.search(OTHER_ACTION, title, re.I) or actions == {"remove"}:  # a rule that only removes entries is a delisting
        return "other action (reclassification, delisting, correction, ...)"
    if re.search(r"critical habitat", title, re.I) and (not status or not actions):  # no status words, or no change to the lists
        return "critical habitat only"
    if re.search(r"under the similarity of appearance", title, re.I):
        return "similarity of appearance only"
    flat = re.sub(r"\s+", " ", text)
    if adds and len(flat) < 25000 and re.search(r"National Marine Fisheries Service|NMFS|NOAA Fisheries", flat[:8000]):
        return "administrative copy of an NMFS listing"
    if adds:
        return "new listing"
    return "status re-determination (no new entry)" if status else "other ESA rule"

esa["kind"] = [classify_rule(r, texts[r.document_number]) for r in esa.itertuples()]
esa.kind.value_counts().to_frame("rules")

,rules
kind,
new listing,304
critical habitat only,243
"other action (reclassification, delisting, correction, ...)",215
not a final listing-type rule,146
other ESA rule,22
status re-determination (no new entry),18
administrative copy of an NMFS listing,16
similarity of appearance only,1


Check the corpus against ECOS: every final FWS listing publication since 1994, matched to my documents by date and first page.

In [5]:
ecos_path = DATA / "ecos_species_fr.parquet"
if not ecos_path.exists():
    columns = ";".join(["/species@id,cn,in,sn,status,status_category,listing_date,agency,is_foreign,country,gn,dps,desc",
                        "/species/taxonomy@kingdom,group", "/species/fws_region@region_name",
                        "/species/fedreg_publication@pub_id,pub_title,pub_page,pub_date,pub_type,action_type,action_option"])
    report = fetch("https://ecos.fws.gov/ecp/pullreports/catalog/species/report/species/export",
                   {"format": "json", "columns": columns, "limit": "200000"}).json()
    names = [c["tablePath"].split("/")[-1] + "." + c["id"] for c in report["meta"]["columns"]]
    flat = lambda v: v.get("value") if isinstance(v, dict) else v
    pd.DataFrame([[flat(v) for v in row] for row in report["data"]], columns=names).to_parquet(ecos_path, index=False)
ecos = pd.read_parquet(ecos_path)
ecos.columns = [c.replace("species.", "").replace("fedreg_publication.", "fr_") for c in ecos.columns]
ecos_listings = ecos[(ecos.fr_action_type == "Listing") & (ecos.fr_pub_type == "Final")
                     & ecos.fr_action_option.isin(["Endangered", "Threatened"])].copy()
ecos_listings["fr_date"] = pd.to_datetime(ecos_listings.fr_pub_date, format="%m-%d-%Y")
ecos_listings = ecos_listings[(ecos_listings.fr_date >= "1994-01-01") & ecos_listings.agency.str.contains("FWS")]
ecos_pubs = (ecos_listings.groupby("fr_pub_id")
             .agg(fr_pub_page=("fr_pub_page", "first"), fr_date=("fr_date", "first"),
                  ecos_title=("fr_pub_title", "first"), ecos_species=("id", "nunique")).reset_index())

def page(citation):
    m = re.search(r"FR\s*(\d+)", citation or "")
    return float(m.group(1)) if m else np.nan

def strip_title(t):
    return re.sub(r"(?i)^(ETWP|Endangered and Threatened Wildlife and Plants|Endangered and Threatened Species)[;:]\s*", "", t or "").lower()

def match_publication(pub, docs):
    """The document an ECOS publication refers to: same date, then nearest first page, else a near-identical title."""
    same = docs[docs.publication_date == pub.fr_date.strftime("%Y-%m-%d")]
    if same.empty:
        return None
    gap = (same.citation.map(page) - page(pub.fr_pub_page)).abs()
    if gap.min() <= 3:
        return same.loc[gap.idxmin(), "document_number"]
    similarity = same.title.map(lambda t: difflib.SequenceMatcher(None, strip_title(t), strip_title(pub.ecos_title)).ratio())
    return same.loc[similarity.idxmax(), "document_number"] if similarity.max() > 0.85 else None

ecos_pubs["document_number"] = [match_publication(p, esa) for p in ecos_pubs.itertuples()]
ecos_pubs = ecos_pubs.merge(esa[["document_number", "kind"]], on="document_number", how="left")
ecos_pubs["kind"] = ecos_pubs.kind.fillna("missing from my Federal Register pull")
print(f"ECOS lists {len(ecos_pubs)} final FWS listing publications since 1994 ({ecos_listings.id.nunique()} species)")
ecos_pubs.groupby([ecos_pubs.fr_date.dt.year < 1995, "kind"]).size().unstack(0, fill_value=0).set_axis(["1995-2026", "1994"], axis=1)

ECOS lists 374 final FWS listing publications since 1994 (1030 species)


,1995-2026,1994
kind,,
administrative copy of an NMFS listing,1,0
critical habitat only,1,0
missing from my Federal Register pull,7,32
new listing,305,0
not a final listing-type rule,5,0
"other action (reclassification, delisting, correction, ...)",12,0
status re-determination (no new entry),11,0


Recover the listings ECOS has and my pull doesn't, from GovInfo's daily MODS records.

In [6]:
def govinfo_issue(date):
    """Every document in one daily Federal Register issue, from GovInfo's MODS metadata (cached)."""
    path = DATA / "mods" / f"FR-{date}.xml"
    if not path.exists():
        path.write_text(fetch(f"https://www.govinfo.gov/metadata/pkg/FR-{date}/mods.xml").text)
    docs = []
    for item in re.split(r'<relatedItem[^>]*type="constituent"', path.read_text())[1:]:
        field = lambda pattern: (re.search(pattern, item, re.S) or [None, None])[1]
        docs.append({"document_number": field(r'<identifier type="FR Doc No\.">([^<]+)'),
                     "citation": field(r'<identifier type="FR citation">([^<]+)'),
                     "title": re.sub(r"\s*;?\s*(?:Final Rules?|Proposed Rules?)?\s*DEPARTMENT OF THE INTERIOR\s*$", "",
                                    html.unescape(field(r"<title>(.*?)</title>") or "")),
                     "abstract": html.unescape(field(r"<abstract>(.*?)</abstract>") or ""),
                     "publication_date": date})
    return pd.DataFrame(docs)

def recover(pub):
    """Find an ECOS listing publication in GovInfo's record of that day's issue."""
    issue = govinfo_issue(pub.fr_date.strftime("%Y-%m-%d"))
    issue = issue[issue.title.str.contains("Endangered and Threatened")]
    match = match_publication(pub, issue)
    if match is None and len(issue):  # 1994 records carry no page numbers: compare species words instead
        words = lambda t: set(re.findall(r"[a-z]{4,}", t.lower())) - {"endangered", "threatened", "status", "determination", "wildlife", "plants", "final", "rule"}
        overlap = (issue.title + " " + issue.abstract).map(lambda t: len(words(t) & words(pub.ecos_title)) / max(1, len(words(pub.ecos_title))))
        match = issue.loc[overlap.idxmax(), "document_number"] if overlap.max() >= 0.5 else None
    return issue[issue.document_number == match].assign(source="recovered from GovInfo via ECOS")

missing = ecos_pubs[ecos_pubs.kind == "missing from my Federal Register pull"]
found = {p.fr_pub_id: recover(p) for p in missing.itertuples()}
recovered = pd.concat(found.values(), ignore_index=True).drop_duplicates("document_number")
recovered["action"] = [re.search(r"ACTION:\s*([^\n]+)", govinfo_text(d, p)).group(1).strip()
                       for d, p in zip(recovered.document_number, recovered.publication_date)]
recovered["kind"] = [classify_rule(r, govinfo_text(r.document_number, r.publication_date)) for r in recovered.itertuples()]
# point each ECOS publication at the document recovered for it
ecos_pubs["document_number"] = ecos_pubs.document_number.fillna(ecos_pubs.fr_pub_id.map({k: d.document_number.iloc[0] for k, d in found.items() if len(d)}))
ecos_pubs["kind"] = ecos_pubs.document_number.map(recovered.set_index("document_number").kind).fillna(ecos_pubs.kind)
print(f"{len(missing)} ECOS publications were missing from my pull; I recovered {len(recovered)} documents from GovInfo")
recovered.groupby([recovered.publication_date.str[:4].rename("year"), "kind"]).size().to_frame("documents")

39 ECOS publications were missing from my pull; I recovered 38 documents from GovInfo


documents
year kind                                             
1994 new listing                                    31
     status re-determination (no new entry)          1
1996 new listing                                     5
1997 new listing                                     1

The corpus: new listings only.

In [7]:
esa = pd.concat([esa, recovered], ignore_index=True)
texts.update({d: govinfo_text(d, p) for d, p in zip(recovered.document_number, recovered.publication_date)})
corpus = esa[esa.kind == "new listing"].copy()
corpus["year"] = corpus.publication_date.str[:4].astype(int)
corpus["period"] = pd.cut(corpus.year, [1993, 1999, 2004, 2009, 2014, 2019, 2026],
                          labels=["1994-99", "2000-04", "2005-09", "2010-14", "2015-19", "2020-26"])
print(f"{len(corpus)} listing rules, {corpus.year.min()}-{corpus.year.max()}, "
      f"{sum(len(texts[d]) for d in corpus.document_number) / 1e6:.0f} million characters")
corpus.groupby("period", observed=True).size().to_frame("listing rules").T

341 listing rules, 1994-2026, 51 million characters


period,1994-99,2000-04,2005-09,2010-14,2015-19,2020-26
listing rules,112,48,14,71,37,59


## Climate language

Paragraphs, each tagged with the section of the rule it is in.

In [8]:
SECTIONS = [  # first match wins; a heading only switches the section if it matches one of these
    ("references", r"^(?:References Cited|Literature Cited)"),
    ("comments", r"Comments|Peer Review|Changes (?:From|to) the Proposed|Summary of Changes"),
    ("threats", r"Summary of Factors|Factors Affecting|Biological Status and Threats|Summary of Threats|Threats? (?:Analysis|Assessment)|Analysis of (?:the )?(?:Listing )?Factors|Regulatory and Analytical Framework|Summary of Biological Status|^(?:Other )?(?:Threats|Factors)$"),
    ("determination", r"^(?:Final )?Determination|Status Determination|Determination of (?:Status|Endangered|Threatened)|Conclusion|Significant Portion of (?:Its|the) Range"),
    ("other", r"^Critical Habitat|Summary of Exclusions|Available Conservation Measures|Required Determinations|National Environmental Policy|Paperwork Reduction|Government-to-Government|Regulation Promulgation|List of Subjects|PART 17|Special Rule|Provisions of the 4\(d\)|4\(d\) Rule|Effects of (?:This|the) Rule|Clarity of the Rule|Authors?$|Authority$"),
    ("background", r"^(?:Background|Species Information|Previous Federal Actions?|Taxonomy|(?:Species )?Description|Biology|Life History|Distribution|Habitat|Executive Summary|Supplementary Information)"),
]

def paragraphs(document_number, text):
    """Split a rule into paragraphs (blank lines or indented first lines) and tag each with its section."""
    rows, section = [], "front matter"
    for i, block in enumerate(re.split(r"\n\s*\n|\n(?= {2,}\S)", text)):
        para = re.sub(r"\s+", " ", block).strip()
        if not para:
            continue
        if len(para) < 120 and not para.endswith((".", ",", ";", ":")):  # heading-like line
            for name, pattern in SECTIONS:
                if re.search(pattern, para, re.I):
                    if not (section == "threats" and name == "background"):  # e.g. "Habitat Loss" inside the threats analysis
                        section = name
                    break
        rows.append((document_number, i, section, para))
    return pd.DataFrame(rows, columns=["document_number", "para", "section", "text"])

paras = pd.concat([paragraphs(d, texts[d]) for d in corpus.document_number], ignore_index=True)
print(f"{len(paras):,} paragraphs; {paras.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory")
paras.groupby("section").text.agg(["size", lambda s: s.str.len().sum()]).set_axis(["paragraphs", "characters"], axis=1)

113,945 paragraphs; 54 MB in memory


,paragraphs,characters
section,,
background,21240,10703407
comments,19923,9558876
determination,3686,2182279
front matter,2079,145257
other,38918,9243817
references,871,179478
threats,27228,16343361


Climate-change terms (not the bare word "climate").

In [9]:
CLIMATE = re.compile(
    r"climat\w*[- ]chang\w*|chang\w* (?:in |of )?(?:the )?(?:global |regional )?climat\w*"   # climate change, changing climate
    r"|global(?:ly)? warm\w*|climat\w* warm\w*|warming (?:of the )?climat\w*"                # global warming, climate warming
    r"|greenhouse (?:gas\w*|effect)|Intergovernmental Panel on Climate Change|\bIPCC\b"
    r"|sea[- ]level rise|rising sea[- ]levels?|sea[- ]levels? (?:are |is |will |have |has )?(?:\w+ )?ris\w*", re.I)

hits = paras[paras.text.str.contains(CLIMATE)]
corpus["climate_paragraphs"] = corpus.document_number.map(hits.groupby("document_number").size()).fillna(0).astype(int)
corpus["any_climate_term"] = corpus.climate_paragraphs > 0
corpus.groupby("period", observed=True).agg(rules=("document_number", "size"), with_climate_terms=("any_climate_term", "sum"))

,rules,with_climate_terms
period,,
1994-99,112,13
2000-04,48,8
2005-09,14,5
2010-14,71,64
2015-19,37,34
2020-26,59,59


Decision rule. **threat**: the Service says climate change harms the species now or is likely to (also as an amplifier of other threats). **possible threat**: may/could/potential, no conclusion. **mention only**: words appear but not as a risk to these species. **no climate terms**. Climate change = the ongoing, human-driven change. Real sentences for each:

In [10]:
ABBREVIATIONS = r"(?<!\bU\.S\.)(?<!\be\.g\.)(?<!\bi\.e\.)(?<!\bal\.)(?<!\bSec\.)(?<!\bpp\.)(?<!\bp\.)(?<!\bNo\.)(?<!\bvs\.)(?<!\bspp\.)(?<!\bvar\.)"

def sentence_with(document_number, snippet):
    """The full sentence of a rule that contains `snippet` (whitespace normalized)."""
    text = re.sub(r"\s+", " ", texts[document_number])
    for sentence in re.split(ABBREVIATIONS + r"(?<=[.!?])\s+(?=[A-Z(\[`'\"])", text):
        if snippet in sentence:
            return sentence
    raise ValueError(f"{snippet!r} not found in {document_number}")

EXAMPLES = [  # (rule, words that locate the sentence, how I label it)
    ("2023-26206", "is a threatened species due primarily to", "threat: the main reason for listing"),
    ("E9-24039", "in and of itself to represent", "threat (amplifier): not a threat on its own ..."),
    ("E9-24039", "potentially important supporting role", "... but it makes the main threats worse"),
    ("2014-10049", "has the potential to impact this species", "possible threat"),
    ("00-7145", "There is no evidence to support global warming", "mention only: the Service rejects it"),
    ("03-5076", "Gradual climate change affecting", "mention only: prehistoric climate change"),
    ("06-4299", "suggested that global warming", "mention only: a peer reviewer's idea, not adopted"),
    ("95-832", "possibly natural climatic changes", "judgment call: natural climatic change"),
]
year = corpus.set_index("document_number").year
for doc, snippet, verdict in EXAMPLES:
    print(f"[{doc}, {year[doc]}] {verdict}\n    {sentence_with(doc, snippet)}\n")

[2023-26206, 2023] threat: the main reason for listing
    We have determined that the contiguous U.S. DPS of the North American wolverine is a threatened species due primarily to the ongoing and increasing impacts of climate change and associated habitat degradation and fragmentation.

[E9-24039, 2009] threat (amplifier): not a threat on its own ...
    Due to the uncertainty associated with climate change projections, we do not consider climate change in and of itself to represent a significant threat to L. papilliferum.

[E9-24039, 2009] ... but it makes the main threats worse
    However, we recognize that the severity and scope of the primary threats to L. papilliferum of frequent wildfire and B. tectorum are likely to magnify depending on the realized outcome of climate change within the foreseeable future; thus, we consider climate change as playing a potentially important supporting role in intensifying the primary current threats to the species.

[2014-10049, 2014] possible th

## Reading the rules with a language model

What the model sees: one rule's climate paragraphs with section tags, cut to the text around each term when a paragraph is long, at most 45,000 characters.

In [11]:
PRIORITY = {"determination": 0, "threats": 1, "front matter": 2, "background": 3, "comments": 4, "other": 5, "references": 6}

def excerpt(text, width=1000):
    """The whole paragraph if it is short; otherwise the text around each climate term."""
    if len(text) <= 2.5 * width:
        return text
    spans = []
    for m in CLIMATE.finditer(text):
        start, end = max(0, m.start() - width), min(len(text), m.end() + width)
        if spans and start <= spans[-1][1]:
            spans[-1][1] = end
        else:
            spans.append([start, end])
    return " ... ".join(text[a:b] for a, b in spans)

def passages(document_number, limit=45000):
    """A rule's climate paragraphs, tagged with their section, in document order, trimmed to `limit` characters."""
    rule = hits[hits.document_number == document_number].assign(excerpt=lambda d: d.text.map(excerpt))
    rule["rank"] = rule.section.map(PRIORITY)
    keep = rule.sort_values(["rank", "para"]).assign(total=lambda d: d.excerpt.str.len().cumsum())
    keep = keep[keep.total <= limit].sort_values("para")
    return "\n\n".join(f"[P{r.para} | {r.section}] {r.excerpt}" for r in keep.itertuples())

def kept(document_number):
    """The numbers of the climate paragraphs that fit within the limit."""
    return [int(p) for p in re.findall(r"^\[P(\d+) \|", passages(document_number), flags=re.M)]

per_rule = hits.groupby("document_number").size()
trimmed = per_rule[[len(kept(d)) < n for d, n in per_rule.items()]]
left_out = pd.concat([hits[(hits.document_number == d) & ~hits.para.isin(kept(d))] for d in trimmed.index])
print(f"{len(trimmed)} of {len(per_rule)} rules have more climate text than the limit; {len(left_out)} of {per_rule.sum():,} "
      f"climate paragraphs are left out, by section: {left_out.section.value_counts().to_dict()}; the most climate "
      f"paragraphs: {per_rule.idxmax()} with {per_rule.max()}")
example = passages("2023-26206")
print(f"The 2023 wolverine rule has {per_rule['2023-26206']} climate paragraphs; {len(kept('2023-26206'))} fit "
      f"({len(example):,} characters). The first two:\n")
print("\n\n".join(p[:600] for p in example.split("\n\n")[:2]))

11 of 183 rules have more climate text than the limit; 276 of 2,790 climate paragraphs are left out, by section: {'comments': 166, 'threats': 53, 'background': 46, 'other': 11}; the most climate paragraphs: E8-11105 with 144
The 2023 wolverine rule has 71 climate paragraphs; 40 fit (44,798 characters). The first two:

[P40 | background] The basis for our action. Under the Act, we may determine that a species is an endangered or threatened species because of any of five factors: (A) The present or threatened destruction, modification, or curtailment of its habitat or range; (B) overutilization for commercial, recreational, scientific, or educational purposes; (C) disease or predation; (D) the inadequacy of existing regulatory mechanisms; or (E) other natural or manmade factors affecting its continued existence. We have determined that the contiguous U.S. DPS of the North American wolverine is a threatened sp

[P55 | background] Following publication of the 2013 proposed rules, there was

What the model returns: a label, one evidence sentence copied from the passages, and a rationale.

In [12]:
class ClimateReading(BaseModel):
    """What one listing rule says about climate change."""
    label: Literal["threat", "possible threat", "mention only"] = Field(
        description="threat / possible threat / mention only, following the definitions in the instructions")
    evidence_quote: str = Field(description="One sentence copied exactly from the passages that best supports the label")
    in_determination: bool = Field(description="True only if climate change is named in the Determination section or the summary of why the species is listed")
    species: list[str] = Field(description="Species the Service says climate change threatens (empty unless label is threat)")
    rationale: str = Field(description="One or two sentences explaining the label")

INSTRUCTIONS = """You are reading a final rule in which the U.S. Fish and Wildlife Service lists species under the
Endangered Species Act. You will see every passage of the rule that uses climate-change language, each tagged with the
section it comes from (threats, determination, background, comments, ...). Decide what the Service itself says about
climate change as a reason these species are at risk.

Labels:
- "threat": In its own voice, the Service says that climate change, or an effect it attributes to climate change
  (warming temperatures, changed rainfall or drought, sea-level rise, loss of sea ice or snowpack, ...), is harming the
  species now or is likely to harm it in the future. This counts even when climate change is one of several threats or
  acts by making other threats worse.
- "possible threat": The Service raises climate change as something that may or could affect the species but does not
  conclude that it does: it calls the effect speculative, uncertain, unknown or only potential, and does not rely on it.
- "mention only": Climate-change words appear, but the Service does not treat climate change as a risk to these species.
  For example: prehistoric or geologic climate change; natural history; a commenter or peer reviewer raised it and the
  Service did not agree; the Service says there is no evidence it is a threat; titles of cited studies; policy text.

Rules:
- Only the Service's own statements count. What commenters or peer reviewers said does not count unless the Service
  agrees in its response.
- Natural weather events (a drought, a hurricane) are not climate change unless the Service links them to climate change.
- evidence_quote must be one sentence copied character for character from the passages."""

`deepseek-v4-flash` on NRP, structured output through LangChain function calling, answers cached in `results/`.

In [13]:
nrp = dict(base_url="https://ellm.nrp-nautilus.io/v1", api_key=os.environ["NRP_TOKEN"])
llm = ChatOpenAI(model="deepseek-v4-flash", temperature=0, timeout=180,
                 extra_body={"thinking": {"type": "disabled"}}, **nrp)
reader = llm.with_structured_output(ClimateReading, method="function_calling", include_raw=True)

def prompt_for(row):
    return f"Rule: {row.title} ({row.publication_date})\n\nPassages:\n\n{passages(row.document_number)}"

def read_rule(row, model=reader):
    """One ClimateReading for one rule, with timing and token counts."""
    prompt = prompt_for(row)
    start = time.time()
    out = model.invoke([SystemMessage(INSTRUCTIONS), HumanMessage(prompt)])
    usage = out["raw"].usage_metadata or {}
    parsed = out["parsed"].model_dump() if out["parsed"] is not None else {"label": None}
    return {"document_number": row.document_number, **parsed, "seconds": round(time.time() - start, 2),
            "input_tokens": usage.get("input_tokens"), "output_tokens": usage.get("output_tokens"),
            "prompt_characters": len(prompt)}

def read_all(rows, path, model=reader, workers=4):
    """Read every rule whose current prompt is not yet in the JSON-lines cache at `path`, four at a time. Each run has its
    own file; it stores each prompt's length, so a rule whose passages change is read again."""
    current = {r.document_number: len(prompt_for(r)) for r in rows.itertuples()}
    done = {(row["document_number"], row.get("prompt_characters")) for row in map(json.loads, open(path))} if Path(path).exists() else set()
    todo = [r for r in rows.itertuples() if (r.document_number, current[r.document_number]) not in done]
    def safe(row):
        for attempt in range(3):
            try:
                return read_rule(row, model)
            except Exception as error:
                last = error
                time.sleep(10)
        return {"document_number": row.document_number, "label": None, "error": str(last)[:300]}
    with open(path, "a") as out, ThreadPoolExecutor(workers) as pool:
        for result in pool.map(safe, todo):
            out.write(json.dumps(result) + "\n")
            out.flush()
    cached = pd.read_json(path, lines=True)
    cached = cached[cached.document_number.map(current) == cached.prompt_characters]
    return cached.drop_duplicates("document_number", keep="last").reset_index(drop=True)

Pilot on 12 rules that span the hard cases (instructions v1).

In [14]:
PILOT = ["95-832", "98-29303", "00-7145", "00-29423", "02-14730", "03-5076", "06-4299",  # 1995-2006: hedges, rejections,
         "E8-11105", "2015-07069", "2016-23112", "2020-09153", "2023-26206"]                   # natural climate; 2008 on: threats
pilot = read_all(corpus[corpus.document_number.isin(PILOT)], RESULTS / "llm_pilot_v1.jsonl")
print(f"{len(pilot)} rules, {pilot.seconds.sum():.0f} seconds of model time, longest {pilot.seconds.max():.0f} s "
      f"({pilot.output_tokens.max():,} output tokens)")
pilot.merge(corpus[["document_number", "year"]], on="document_number").sort_values("year")[["year", "document_number", "label", "evidence_quote"]]

12 rules, 208 seconds of model time, longest 38 s (3,455 output tokens)


,year,document_number,label,evidence_quote
0,1995,95-832,possible threat,"It has declined because of human bird and egg collectors (B), predation by introduced rats (C), and possibly natural..."
1,1998,98-29303,possible threat,Climate changes (either natural or human-caused) are also a potential threat to the species.
2,2000,00-7145,mention only,There is no evidence to support global warming as a threat to the lynx.
3,2000,00-29423,possible threat,"Global warming can have an effect on sea temperatures, wind currents, fresh water input, and mixing of the ocean's s..."
4,2002,02-14730,possible threat,A number of other factors have been identified as causes or possible causes of global amphibian decline; although th...
5,2003,03-5076,mention only,Gradual climate change affecting the distribution and composition of sagebrush communities is thought to have result...
6,2006,06-4299,mention only,The reviewer noted that at least 3 of the 12 flies proposed for listing have demonstrated an apparent habitat shift ...
7,2008,E8-11105,threat,We have evaluated the best available scientific and commercial information on polar bear habitat and the current and...
8,2015,2015-07069,possible threat,"Although we have information that suggests that climate change may affect the northern long-eared bat, we do not hav..."
9,2016,2016-23112,threat,The effects of rising temperature and other aspects of climate change are likely to exacerbate many of these threats...
